### Notebook 范围

### 目的
重绘 Hindcast 多 case 的 O3、U60N10、U60N50 evolution，并把同年份、同初始化月、同扰动类型的 INT 与 NOCOUPL 放到同一张图上比较。

### 科学问题
H-INT-3D 和 H Clim 3D 的臭氧路径、10 hPa 极涡风速路径、50 hPa 极涡风速路径是否系统性分离？这种分离是否相对于 BWCN 同年 reference 和 B2000 气候态有物理意义？

### 方法
O3 使用 60-90N、30-70 hPa partial column；U 使用 60N zonal mean，分别插值到 10 hPa 和 50 hPa。彩色细线是成员，彩色粗线和阴影是 ensemble mean ±1 std，黑实线是 BWCN 同年 reference，黑点线是 B2000 climatology。x 轴统一为 Jan-May 的月份坐标，所有同类三联图使用统一 xlim 和 ylim。

### 预期输出
单 case 图写入 `outputs/figures/02_evolution/by_case/<year>/`；INT vs NOCOUPL 对比图写入 `outputs/figures/02_evolution/paired_INT_vs_NOCOUPL/<year>/`。每张图同时保存 png/pdf，并有同名 csv 支撑表格。

### 解读
如果 H-INT-3D 与 H Clim 3D 的 ensemble mean 或 spread 在 U60N10/U60N50 上先分离，再在 O3 上分离，说明 interactive O3 影响的是 wave-vortex-ozone pathway，而不只是臭氧诊断本身。

### 注意
0003 没有强行配 NOCOUPL；v2/v3 如果没有同类 NOCOUPL，只画单 case 图。later-initialized cases 缺少初始化前月份，不能解释 long-lead precursor，只能解释初始化后路径差异。


### 导入与公共路径

### 目的
加载公共工具函数、路径和绘图依赖。

### 科学问题
这一步不直接回答科学问题，但保证所有图使用同一套 cleaned hindcast 产品和同一套变量定义。

### 方法
从 `hindcast_extension_utils.py` 读取数据路径、case 发现函数、O3/U/reference/climatology 加载函数，以及统一保存图和记录 figure manifest 的工具。

### 预期输出
打印工作目录和 Hindcast 数据目录，不产生图。

### 解读
路径正确说明后续代码会从 `/mnt/soclim0/public_data/weiji/Hindcast` 读数据，并只把输出写到 `code_cleaned/Hindcast_experiment/Extention_analysis/outputs`。

### 注意
Notebook 只读原始数据；不会修改 `/mnt/soclim0/public_data/weiji` 下的任何文件。


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-code-cleaned")

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 统一绘图风格和辅助函数

### 目的
定义接近 EGU poster 的 evolution 绘图风格，并实现成员、ensemble mean、reference、climatology 的统一绘制函数。

### 科学问题
统一样式本身不回答科学问题，但它避免不同 case 因坐标范围或图例差异造成视觉误判。

### 方法
H-INT-3D 使用绿色，H Clim 3D 使用蓝色；v2 大扰动在图例中标为 H INT-3D LP，v3 湿度扰动标为 H INT-3D MP。reference 使用黑实线，climatology 使用黑点线。所有图使用 Jan-May 的统一月份坐标，O3/U 的 ylim 由所有 case、reference 和 climatology 全局扫描后确定。

### 预期输出
本代码块只定义函数和样式，不保存图。

### 解读
后续每张图中，彩色粗线之间的距离表示 INT 与 NOCOUPL 的 ensemble mean 差异；阴影表示 ensemble spread；黑线和点线用于判断路径相对真实事件和气候态的位置。

### 注意
全局 ylim 是为了跨 case 可比性，不是为了突出单个 case 的局部细节；极端 outlier 会扩大坐标范围。


In [ ]:
VERSION_TAG = "v5"
X_START_DOY = 1
X_END_DOY = mmdd_to_doy(5, 30)

STYLE = {
    "int_color": "forestgreen",
    "nocoupl_color": "royalblue",
    "v2_color": "darkorange",
    "v3_color": "purple",
    "mean_lw": 3.4,
    "member_lw": 0.9,
    "member_alpha": 0.22,
    "sigma_alpha": 0.28,
    "ref_lw": 3.0,
    "clim_lw": 2.6,
    "grid_alpha": 0.30,
}

VAR_SPECS = {
    "O3": {
        "ylabel": "O3 partial column (DU)\n60-90N, 30-70 hPa",
        "title": "Arctic O3 partial column",
        "step": 10,
    },
    "U60N1": {
        "ylabel": "U60N1 (m s$^{-1}$)",
        "title": "Polar vortex wind at 1 hPa",
        "step": 10,
    },
    "U60N10": {
        "ylabel": "U60N10 (m s$^{-1}$)",
        "title": "Polar vortex wind at 10 hPa",
        "step": 10,
    },
    "U60N50": {
        "ylabel": "U60N50 (m s$^{-1}$)",
        "title": "Polar vortex wind at 50 hPa",
        "step": 10,
    },
}

MONTH_TICKS_FOR_PLOT = [1, 32, 60, 91, 121]
MONTH_LABELS_FOR_PLOT = ["Jan", "Feb", "Mar", "Apr", "May"]


def _experiment_label(row):
    if row["config"] == "NOCOUPL":
        return "H Clim 3D"
    if row["perturbation"] == "v2_large_temperature":
        return "H INT-3D LP"
    if row["perturbation"] == "v3_humidity":
        return "H INT-3D MP"
    return "H INT-3D"


def _experiment_color(row):
    if row["config"] == "NOCOUPL":
        return STYLE["nocoupl_color"]
    if row["perturbation"] == "v2_large_temperature":
        return STYLE["v2_color"]
    if row["perturbation"] == "v3_humidity":
        return STYLE["v3_color"]
    return STYLE["int_color"]


def _safe_name(text):
    return str(text).replace("/", "_").replace(" ", "_").replace("+", "plus")


def _set_month_axis(ax):
    ax.set_xlim(X_START_DOY, X_END_DOY)
    ax.set_xticks(MONTH_TICKS_FOR_PLOT)
    ax.set_xticklabels(MONTH_LABELS_FOR_PLOT)
    ax.set_xlabel("Month")


def _member_mean_std(da):
    return da.mean("member", skipna=True), da.std("member", skipna=True)


def _case_x(case, date, n):
    return case_time_doy(case, np.asarray(date)[:n])


def _within_plot_window(x):
    x = np.asarray(x, dtype=float)
    return (x >= X_START_DOY) & (x <= X_END_DOY)


def _finite_values(values):
    if isinstance(values, (list, tuple)):
        parts = []
        for item in values:
            arr = np.asarray(item, dtype=float).ravel()
            arr = arr[np.isfinite(arr)]
            if arr.size:
                parts.append(arr)
        if not parts:
            return np.asarray([], dtype=float)
        return np.concatenate(parts)
    arr = np.asarray(values, dtype=float).ravel()
    return arr[np.isfinite(arr)]


def _nice_ylim(values, step=10, pad_frac=0.07, fallback=(-1, 1)):
    vals = _finite_values(values)
    if vals.size == 0:
        return fallback
    lo = float(np.nanmin(vals))
    hi = float(np.nanmax(vals))
    if not np.isfinite(lo) or not np.isfinite(hi):
        return fallback
    if math.isclose(lo, hi):
        lo -= step
        hi += step
    pad = max((hi - lo) * pad_frac, step * 0.5)
    return (math.floor((lo - pad) / step) * step, math.ceil((hi + pad) / step) * step)


def _plot_experiment(ax, da, date, case, color, label, table_rows, variable):
    if da is None:
        return
    da = da.transpose("member", "lead_time")
    n = da.sizes["lead_time"]
    x = _case_x(case, date, n)
    mask = _within_plot_window(x)
    if not mask.any():
        return
    x = x[mask]
    sub = da.isel(lead_time=np.where(mask)[0])
    for i in range(sub.sizes["member"]):
        ax.plot(x, sub.isel(member=i).values, color=color, alpha=STYLE["member_alpha"], lw=STYLE["member_lw"], zorder=2)
    mean, std = _member_mean_std(sub)
    lower = mean - std
    upper = mean + std
    ax.fill_between(x, lower.values, upper.values, color=color, alpha=STYLE["sigma_alpha"], linewidth=0, zorder=1)
    ax.plot(x, lower.values, color=color, lw=0.8, alpha=0.85, zorder=2)
    ax.plot(x, upper.values, color=color, lw=0.8, alpha=0.85, zorder=2)
    ax.plot(x, mean.values, color=color, lw=STYLE["mean_lw"], label=label, zorder=4)
    for d, m, s in zip(x, mean.values, std.values):
        table_rows.append({
            "case": case,
            "source": label,
            "variable": variable,
            "doy": int(d),
            "ensemble_mean": float(m) if np.isfinite(m) else np.nan,
            "ensemble_std": float(s) if np.isfinite(s) else np.nan,
            "value": np.nan,
        })


def _plot_reference(ax, ref, date, year, table_rows, variable):
    if ref is None or date is None:
        return
    n = ref.sizes["lead_time"]
    x = case_time_doy(f"{str(year).zfill(4)}-01", np.asarray(date)[:n])
    mask = _within_plot_window(x)
    if not mask.any():
        return
    x = x[mask]
    y = ref.isel(lead_time=np.where(mask)[0]).values
    ax.plot(x, y, color="black", lw=STYLE["ref_lw"], label="Reference", zorder=10)
    for d, val in zip(x, y):
        table_rows.append({
            "case": f"BWCN_{str(year).zfill(4)}",
            "source": "Reference",
            "variable": variable,
            "doy": int(d),
            "ensemble_mean": np.nan,
            "ensemble_std": np.nan,
            "value": float(val) if np.isfinite(val) else np.nan,
        })


def _plot_climatology(ax, clim, table_rows, variable):
    if clim is None:
        return
    sub = clim.sel(doy=slice(X_START_DOY, X_END_DOY))
    x = np.asarray(sub["doy"].values, dtype=int)
    y = sub.values
    ax.plot(x, y, color="black", ls=":", lw=STYLE["clim_lw"], label="Climatology", zorder=9)
    for d, val in zip(x, y):
        table_rows.append({
            "case": "B2000WCN001002",
            "source": "Climatology",
            "variable": variable,
            "doy": int(d),
            "ensemble_mean": np.nan,
            "ensemble_std": np.nan,
            "value": float(val) if np.isfinite(val) else np.nan,
        })


def _format_axes(axes, variables, axis_limits):
    for ax, variable in zip(axes, variables):
        if variable in axis_limits:
            ax.set_ylim(axis_limits[variable])
        if variable.startswith("U60N"):
            label = "0 m s$^{-1}$" if variable == "U60N1" else None
            ax.axhline(0, color="0.3", lw=0.8, zorder=0, label=label)
        if variable == "U60N50":
            ax.axhline(U60_THRESH, color="tab:red", lw=1.1, ls="--", label="50 hPa FWD threshold: 7 m s$^{-1}$", zorder=0)
        ax.grid(True, alpha=STYLE["grid_alpha"], linestyle="--")
        ax.tick_params(labelsize=10)
        _set_month_axis(ax)


def _dedupe_legend(ax):
    handles, labels = ax.get_legend_handles_labels()
    out_h, out_l = [], []
    for h, l in zip(handles, labels):
        if l not in out_l:
            out_h.append(h); out_l.append(l)
    ax.legend(out_h, out_l, loc="best", fontsize=8.5, frameon=True, ncol=2)


### 加载所有 evolution 数据并计算统一坐标范围

### 目的
一次性加载所有 case 的 O3、U60N10、U60N50、BWCN reference 和 B2000 climatology，并计算全局统一的 ylim。

### 科学问题
这一步为跨年份、跨 case 的公平视觉比较做准备：同类变量在所有图上必须使用同一坐标范围。

### 方法
扫描 Hindcast 目录；对每个 case 读取 O3 partial column 和 U60N pressure-level time series；对每个 reference year 读取 BWCN O3/U；对 climatology 读取 B2000 O3/U。ylim 使用所有 Jan-May 内的有限值加 padding 后取整。

### 预期输出
保存 `outputs/tables/02_evolution/02_evolution_axis_limits_v3.csv`，记录 O3、U60N10、U60N50 的统一 y 轴范围。

### 解读
如果某个变量坐标范围很宽，说明至少一个 case/reference/climatology 中存在较强异常或较大 ensemble spread。

### 注意
统一 ylim 会牺牲单个 case 的局部放大效果；这是为了避免 INT/NOCOUPL 或不同年份之间的视觉尺度偏差。


In [ ]:
inv = discover_hindcast_cases()
print(inv[["case", "year", "init_month", "config", "perturbation", "n_members"]].to_string(index=False))

case_data = {}
ref_data = {}

clim_data = {
    "O3": load_b2000_o3_partial_climatology(),
    "U60N1": load_b2000_u60_climatology(1),
    "U60N10": load_b2000_u60_climatology(10),
    "U60N50": load_b2000_u60_climatology(50),
}

small_cases = inv[inv["perturbation"] == "small_temperature"].copy()
paired_cases_for_u1 = set()
paired_years_for_u1 = set()
for _, int_row in small_cases[small_cases["config"] == "INT"].iterrows():
    match = small_cases[
        (small_cases["year"] == int_row["year"])
        & (small_cases["init_month"] == int_row["init_month"])
        & (small_cases["config"] == "NOCOUPL")
    ]
    if match.empty:
        continue
    paired_cases_for_u1.add(int_row["case"])
    paired_cases_for_u1.add(match.iloc[0]["case"])
    paired_years_for_u1.add(str(int_row["year"]).zfill(4))

for _, row in inv.iterrows():
    case = row["case"]
    o3, o3_date = load_hindcast_o3(case)
    u10, u10_date = compute_u60(case, 10)
    u50, u50_date = compute_u60(case, 50)
    bundle = {
        "row": row.to_dict(),
        "O3": (o3, o3_date),
        "U60N10": (u10, u10_date),
        "U60N50": (u50, u50_date),
    }
    if case in paired_cases_for_u1:
        u1, u1_date = compute_u60(case, 1)
        bundle["U60N1"] = (u1, u1_date)
    case_data[case] = bundle

for year in sorted(inv["year"].unique()):
    year_key = str(year).zfill(4)
    ref_o3, ref_o3_date = load_bwcn_reference_o3(year)
    ref_u10, ref_u10_date = load_bwcn_reference_u60(year, 10)
    ref_u50, ref_u50_date = load_bwcn_reference_u60(year, 50)
    bundle = {
        "O3": (ref_o3, ref_o3_date),
        "U60N10": (ref_u10, ref_u10_date),
        "U60N50": (ref_u50, ref_u50_date),
    }
    if year_key in paired_years_for_u1:
        ref_u1, ref_u1_date = load_bwcn_reference_u60(year, 1)
        bundle["U60N1"] = (ref_u1, ref_u1_date)
    ref_data[year_key] = bundle

limit_values = {"O3": [], "U60N1": [], "U60N10": [], "U60N50": []}
for case, bundle in case_data.items():
    for variable in limit_values:
        if variable not in bundle:
            continue
        da, date = bundle[variable]
        if da is None:
            continue
        x = _case_x(case, date, da.sizes["lead_time"])
        mask = _within_plot_window(x)
        if mask.any():
            limit_values[variable].append(da.isel(lead_time=np.where(mask)[0]).values)

for year, bundle in ref_data.items():
    for variable in limit_values:
        if variable not in bundle:
            continue
        da, date = bundle[variable]
        if da is None:
            continue
        x = case_time_doy(f"{str(year).zfill(4)}-01", date[: da.sizes["lead_time"]])
        mask = _within_plot_window(x)
        if mask.any():
            limit_values[variable].append(da.isel(lead_time=np.where(mask)[0]).values)

for variable, clim in clim_data.items():
    if variable in limit_values and clim is not None:
        limit_values[variable].append(clim.sel(doy=slice(X_START_DOY, X_END_DOY)).values)

axis_limits = {
    variable: _nice_ylim(values, step=VAR_SPECS[variable]["step"])
    for variable, values in limit_values.items()
}

limit_table = pd.DataFrame([
    {"variable": variable, "ymin": ylim[0], "ymax": ylim[1], "x_start_doy": X_START_DOY, "x_end_doy": X_END_DOY}
    for variable, ylim in axis_limits.items()
])
limit_csv = table_dir("02_evolution") / f"02_evolution_axis_limits_{VERSION_TAG}.csv"
limit_table.to_csv(limit_csv, index=False)
print(limit_table.to_string(index=False))


### 绘制单 case evolution 图

### 目的
为每个 hindcast case 生成一张横向三联图，作为基本路径检查：O3、U60N10、U60N50。

### 科学问题
每个 case 的 ensemble pathway 相对于同年 BWCN reference 和 B2000 climatology 偏向哪里？v2/v3 或缺少 NOCOUPL 配对的 case 是否仍然表现出明显的 vortex/O3 spread？

### 方法
每张图包含一个 experiment 的所有成员、ensemble mean ±1 std、同年 BWCN reference、B2000 climatology。三联图按 EGU poster 风格横向排列，所有图统一 Jan-May xlim，且 O3/U 使用上一步全局统一 ylim。

### 预期输出
`outputs/figures/02_evolution/by_case/<year>/02_evolution_<case>_O3_U60N10_U60N50_v3.png/pdf`，以及 `outputs/tables/02_evolution/by_case/<year>/` 下同名 csv。

### 解读
彩色 ensemble mean 靠近黑实线表示接近同年 reference；靠近黑点线表示更接近 climatology。U60N50 跨过 7 m/s 阈值的时间用于理解 final warming timing。

### 注意
单 case 图不能直接说明 interactive O3 feedback；feedback 需要看下一块的 INT vs NOCOUPL 同图比较。


In [ ]:
def _get_case_variable(case, variable):
    bundle = case_data.setdefault(case, {"row": {"case": case}})
    if variable not in bundle and variable.startswith("U60N"):
        plev = int(variable.replace("U60N", ""))
        bundle[variable] = compute_u60(case, plev)
    return bundle.get(variable, (None, None))


def _get_reference_variable(year, variable):
    year_key = str(year).zfill(4)
    bundle = ref_data.setdefault(year_key, {})
    if variable not in bundle and variable.startswith("U60N"):
        plev = int(variable.replace("U60N", ""))
        bundle[variable] = load_bwcn_reference_u60(year_key, plev)
    return bundle.get(variable, (None, None))


def _get_climatology_variable(variable):
    if variable not in clim_data and variable.startswith("U60N"):
        plev = int(variable.replace("U60N", ""))
        clim_data[variable] = load_b2000_u60_climatology(plev)
    return clim_data.get(variable)


def plot_evolution_figure(
    experiments,
    year,
    title,
    fig_dir,
    tab_dir,
    filename,
    guide_question,
    guide_interpretation,
    variables=None,
    layout=(1, 3),
    figsize=(18.0, 5.8),
):
    if variables is None:
        variables = ["O3", "U60N10", "U60N50"]
    fig, axes = plt.subplots(layout[0], layout[1], figsize=figsize, sharex=True, constrained_layout=True)
    axes_flat = np.asarray(axes).ravel()
    table_rows = []

    for ax, variable in zip(axes_flat, variables):
        for exp in experiments:
            row = exp["row"]
            case = row["case"]
            da, date = _get_case_variable(case, variable)
            if da is None:
                log_message(f"{case}: missing {variable} for 02 evolution")
                continue
            _plot_experiment(
                ax=ax,
                da=da,
                date=date,
                case=case,
                color=exp["color"],
                label=exp["label"],
                table_rows=table_rows,
                variable=variable,
            )
        ref, ref_date = _get_reference_variable(year, variable)
        _plot_reference(ax, ref, ref_date, year, table_rows, variable)
        _plot_climatology(ax, _get_climatology_variable(variable), table_rows, variable)
        ax.set_ylabel(VAR_SPECS[variable]["ylabel"])
        ax.set_title(VAR_SPECS[variable]["title"], fontsize=12)

    for ax in axes_flat[len(variables):]:
        ax.axis("off")
    _format_axes(axes_flat[:len(variables)], variables, axis_limits)
    for ax in axes_flat[:len(variables)]:
        _dedupe_legend(ax)
    fig.suptitle(title, fontsize=15, fontweight="bold")

    table = pd.DataFrame(table_rows)
    csv = tab_dir / f"{filename}.csv"
    table.to_csv(csv, index=False)
    u_levels = "/".join(v.replace("U60N", "") for v in variables if v.startswith("U60N"))
    savefig(
        fig,
        filename,
        fig_dir=fig_dir,
        notebook="02_multicase_O3_U_evolution.ipynb",
        scientific_question=guide_question,
        variables_windows=f"O3: 60-90N 30-70hPa; U: zonal-mean 60N at {u_levels}hPa; x=Jan-May; unified ylim across all 02 evolution figures; black=BWCN same-year reference; dotted=B2000 climatology.",
        interpretation=guide_interpretation,
        caveat="Later-initialized cases do not contain pre-initialization hindcast data; those months only show reference/climatology. Paired plots include 1 hPa wind for upper-stratospheric context.",
        csv_table=csv,
    )
    plt.close(fig)
    return csv

single_summary_rows = []
for _, row in inv.iterrows():
    rowd = row.to_dict()
    case = rowd["case"]
    year = rowd["year"]
    exp = {
        "row": rowd,
        "label": _experiment_label(rowd),
        "color": _experiment_color(rowd),
    }
    fig_dir = figure_dir("02_evolution", "by_case", year)
    tab_dir = table_dir("02_evolution", "by_case", year)
    filename = f"02_evolution_{_safe_name(case)}_O3_U60N10_U60N50_{VERSION_TAG}"
    csv = plot_evolution_figure(
        experiments=[exp],
        year=year,
        title=f"{case}: O3 and vortex wind evolution",
        fig_dir=fig_dir,
        tab_dir=tab_dir,
        filename=filename,
        guide_question="单个 hindcast case 的 O3/U pathway 是否接近 BWCN reference 或 B2000 climatology？",
        guide_interpretation="彩色 ensemble mean 与 reference/climatology 的相对位置用于判断该 case 的动力-臭氧路径偏差。",
        variables=["O3", "U60N10", "U60N50"],
        layout=(1, 3),
        figsize=(18.0, 5.8),
    )
    single_summary_rows.append({"case": case, "year": year, "figure": str(fig_dir / f"{filename}.png"), "table": str(csv)})

single_summary = pd.DataFrame(single_summary_rows)
single_summary_csv = table_dir("02_evolution") / f"02_single_case_evolution_summary_{VERSION_TAG}.csv"
single_summary.to_csv(single_summary_csv, index=False)
print(single_summary.to_string(index=False))


### 绘制 H-INT-3D 与 H Clim 3D 同类配对 evolution 图

### Purpose

这一块把同一年、同初始化月份、同为 small-temperature perturbation 的 H-INT-3D 与 H Clim 3D 画在同一张图里。paired 图使用 2×2 排布，变量为 O3、U60N1、U60N10 和 U60N50。

### Scientific question

interactive O3 feedback 是否同时改变臭氧柱、上平流层 1 hPa 风、10 hPa polar vortex 和 50 hPa breakdown pathway？

### Method

彩色细线为 members，粗线为 ensemble mean，阴影为 ±1 std；黑实线为同年 BWCN reference，黑点线为 B2000 climatology。所有 paired 图使用统一 y 轴范围，x 轴用月份坐标。

### Expected output

输出到 `outputs/figures/02_evolution/paired_INT_vs_NOCOUPL/<year>/`，支撑 CSV 输出到对应 `outputs/tables/02_evolution/paired_INT_vs_NOCOUPL/<year>/`。

### Interpretation

如果 H-INT-3D 与 H Clim 3D 的 mean/spread 在 U60N1、U60N10 或 U60N50 上分离，说明 ozone feedback 可能影响 vortex vertical structure 和 final warming pathway。

### Caveat

这仍是 ensemble-level 对比，不强行要求 member 一一对应。later initialized cases 的初始化前月份没有 hindcast 成员线，只能看 reference/climatology。


In [ ]:
pair_summary_rows = []
small = inv[inv["perturbation"] == "small_temperature"].copy()
for _, int_row in small[small["config"] == "INT"].iterrows():
    year = int_row["year"]
    init_month = int_row["init_month"]
    match = small[
        (small["year"] == year)
        & (small["init_month"] == init_month)
        & (small["config"] == "NOCOUPL")
    ]
    if match.empty:
        msg = f"{int_row['case']}: no matching NOCOUPL case for paired 02 evolution"
        log_message(msg)
        pair_summary_rows.append({"int_case": int_row["case"], "nocoupl_case": "", "year": year, "init_month": init_month, "status": "missing_NOCOUPL"})
        continue
    noc_row = match.iloc[0]
    int_exp = {"row": int_row.to_dict(), "label": "H INT-3D", "color": STYLE["int_color"]}
    noc_exp = {"row": noc_row.to_dict(), "label": "H Clim 3D", "color": STYLE["nocoupl_color"]}
    fig_dir = figure_dir("02_evolution", "paired_INT_vs_NOCOUPL", year)
    tab_dir = table_dir("02_evolution", "paired_INT_vs_NOCOUPL", year)
    filename = f"02_evolution_pair_{_safe_name(int_row['case'])}_vs_{_safe_name(noc_row['case'])}_O3_U60N1_U60N10_U60N50_{VERSION_TAG}"
    csv = plot_evolution_figure(
        experiments=[int_exp, noc_exp],
        year=year,
        title=f"{int_row['case']} vs {noc_row['case']}: H INT-3D vs H Clim 3D",
        fig_dir=fig_dir,
        tab_dir=tab_dir,
        filename=filename,
        guide_question="同年同初始化的 H INT-3D 和 H Clim 3D 是否表现出不同 O3/U pathway？",
        guide_interpretation="绿色和蓝色的 ensemble mean/spread 分离表示 interactive O3 feedback 可能改变 upper/lower-stratospheric vortex persistence 或臭氧损耗路径。",
        variables=["O3", "U60N1", "U60N10", "U60N50"],
        layout=(2, 2),
        figsize=(13.8, 9.4),
    )
    pair_summary_rows.append({
        "int_case": int_row["case"],
        "nocoupl_case": noc_row["case"],
        "year": year,
        "init_month": init_month,
        "status": "plotted",
        "figure": str(fig_dir / f"{filename}.png"),
        "table": str(csv),
    })

pair_summary = pd.DataFrame(pair_summary_rows)
pair_summary_csv = table_dir("02_evolution") / f"02_paired_INT_vs_NOCOUPL_evolution_summary_{VERSION_TAG}.csv"
pair_summary.to_csv(pair_summary_csv, index=False)

combined_summary = pd.concat([
    single_summary.assign(kind="single_case"),
    pair_summary.rename(columns={"int_case": "case"}).assign(kind="paired_INT_vs_NOCOUPL"),
], ignore_index=True, sort=False)
summary_csv = table_dir("02_evolution") / f"02_multicase_evolution_summary_{VERSION_TAG}.csv"
combined_summary.to_csv(summary_csv, index=False)
# Keep the historical root-level filename available for downstream notebooks.
combined_summary.to_csv(TAB_DIR / "02_multicase_evolution_summary.csv", index=False)

print(pair_summary.to_string(index=False))
write_figure_guide()
